# omnisus no Colab: do DATASUS a uma taxa citável

Este notebook percorre a biblioteca de ponta a ponta com uma UF pequena, Roraima:

- os óbitos de 2022 (SIM) e as AIH processadas em janeiro de 2023 (SIH);
- a população do censo 2022 do IBGE.

Ele lista as bases, baixa os arquivos para um lake, confere as colunas e põe rótulos nos
códigos. Depois usa as categorias harmonizadas de idade e sexo, agrupa as causas pela
CID-10, calcula óbitos por 100 mil habitantes e cita cada arquivo usado.

Rode as células em ordem (**Ambiente de execução → Executar tudo** também funciona). O
download tem poucos MB e leva alguns minutos.

> **Trabalho em construção.** Confira as contagens com o DATASUS e os rótulos com a
> fonte antes de publicar: [como conferir](https://raphaelfh.github.io/omnisus/dicionario/).

## 1. Instalar

Instala com o `uv`, que já vem no Colab, a versão do PyPI fixada abaixo.

In [ ]:
!uv pip install -q "omnisus[colab]==0.2.3"

## 2. Parâmetros e lake

Troque a UF e os anos aqui. Sem o Drive, o lake fica em `/content/data/raw` e some
quando o ambiente do Colab é desligado. Com `USAR_DRIVE = True`, ele fica no seu Google
Drive, e um novo `load` do mesmo escopo não baixa nada de novo.

As categorias harmonizadas (seção 8) só existem para as fontes validadas que a tabela
da seção 4 lista. Fora delas, `load` avisa que as deixou de fora, e a seção 8 diz isso
em vez de calcular.

In [ ]:
import polars as pl

import omnisus as sus

# Até 25 linhas e 80 caracteres por célula em cada tabela.
pl.Config.set_tbl_rows(25).set_fmt_str_lengths(80)

UF = "RR"
ANO_SIM = 2022  # a seção 11 divide os óbitos deste ano pela população do censo 2022
ANO_SIH, MES_SIH = 2023, 1
USAR_DRIVE = False

if USAR_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    sus.set_lake_dir("/content/drive/MyDrive/omnisus")

## 3. Que bases existem

`describe_datasets` lista, sem rede, as bases do FTP do DATASUS que o omnisus importa:
o nome que vai em `load`, o sistema, o que é, se há um arquivo por UF (`state`) ou um
nacional (`national`), se é mensal e a cobertura declarada. A página
[Bases e argumentos](https://raphaelfh.github.io/omnisus/datasets/) traz a mesma lista,
por sistema.

In [ ]:
# Todas as linhas, só nesta tabela.
with pl.Config(tbl_rows=-1):
    display(
        sus.describe_datasets().select(
            "name", "category", "title", "geography", "cadence", "coverage_start", "coverage_end"
        )
    )

## 4. O que uma base registra

`describe_dataset` traz o dicionário de uma base, sem rede: cada campo com o tipo e o
rótulo, e as fontes validadas, os arquivos do servidor (pelo SHA-256) cujos códigos
foram conferidos contra a regra das categorias harmonizadas. Antes de interpretar
números, leia o [perfil do SIM](https://raphaelfh.github.io/omnisus/sources/sim_obitos/).

In [ ]:
sim = sus.describe_dataset("sim_obitos")
display(
    pl.DataFrame(
        [
            {"campo": f["name"], "tipo": f["type"], "rótulo": f.get("label", "")}
            for f in sim["schema"]["fields"]
        ]
    ).head(10)
)
pl.DataFrame(sim["analytics"]["validated_sources"])

## 5. O que o DATASUS publica

`available` consulta o servidor e lista os escopos que existem: no SIM, um arquivo por
UF e ano; no SIH, um por UF e mês de processamento.

In [ ]:
display(sus.available("sim_obitos", years=[ANO_SIM], ufs=[UF]))
sus.available("sih_aih_reduzida", years=[ANO_SIH], ufs=[UF])

## 6. Baixar e ler

`load` importa os escopos para o lake e devolve as linhas, com os códigos como o
DATASUS publicou. Rodar de novo não baixa nem duplica nada. Numa base mensal, passe
`months`; sem ele, `load` importa os doze meses do ano.

In [ ]:
obitos = sus.load("sim_obitos", years=[ANO_SIM], ufs=[UF])
aih = sus.load("sih_aih_reduzida", years=[ANO_SIH], ufs=[UF], months=[MES_SIH])
print(f"{obitos.height} declarações de óbito e {aih.height} linhas de AIH")

## 7. Conferir e pôr rótulos

`check_columns` mostra, por coluna, vazios, códigos sem rótulo e datas fora do esperado.
`outdated` lista os escopos que o DATASUS republicou depois do download. `label`
acrescenta `<coluna>_rotulo` a partir do dicionário; um código que o dicionário não
conhece fica sem rótulo.

In [ ]:
with sus.LakeReader() as lake:
    print("Republicados desde o download:", sus.outdated("sim_obitos", lake=lake) or "nenhum")
sus.check_columns("sim_obitos", obitos)

In [ ]:
rotulados = sus.label("sim_obitos", obitos, columns=["sexo", "racacor"])
rotulados.group_by("racacor", "racacor_rotulo").len("obitos").sort("obitos", descending=True)

## 8. Categorias harmonizadas

Para uma fonte validada, `load` acrescenta colunas que comparam o mesmo conceito entre
bases: `idade_anos_completos` e `idade_status`, `sexo_categoria` e `sexo_status`, e
cada data como `<campo>_data`. Uma idade que a regra não interpreta fica nula, com o
motivo em `idade_status`. Num escopo que não é fonte validada, essas colunas não existem
e a célula abaixo diz isso. Veja
[consumo dos metadados](https://raphaelfh.github.io/omnisus/dicionario/consumo/).

In [ ]:
if "idade_status" not in obitos.columns:
    print("A fonte deste escopo não é validada: load deixou as categorias harmonizadas de fora.")
else:
    display(obitos.group_by("idade_status").len("obitos"))
    display(
        obitos.filter(pl.col("idade_status") == "valid")
        .group_by(
            faixa_etaria=pl.col("idade_anos_completos") // 10 * 10, sexo=pl.col("sexo_categoria")
        )
        .len("obitos")
        .pivot(on="sexo", index="faixa_etaria", values="obitos")
        .fill_null(0)
        .sort("faixa_etaria")
    )
    # Óbitos por mês, pela data do óbito já convertida em data.
    display(obitos.group_by(mes=pl.col("dtobito_data").dt.month()).len("obitos").sort("mes"))

## 9. Causas de óbito pela CID-10

`load` grava só a base pedida. Os vocabulários (CID-10, municípios, UF, CBO e países)
vêm dentro do pacote, e `bootstrap_auxiliares` os grava no lake, sem rede, como tabelas
`aux_*`; sem ele, a `aux_cid10` não existe. `read` devolve uma tabela do lake como
DataFrame polars, e daqui em diante as junções são em polars: a causa básica
(`causabas`) junta com o `codigo` da `aux_cid10`, como o dicionário declara.

In [ ]:
# load grava no lake só a base pedida. A CID-10 e os outros vocabulários vêm dentro
# do pacote, e bootstrap_auxiliares os grava no lake (tabelas aux_*), sem rede.
with sus.Lake.local() as lake:
    lake.bootstrap_auxiliares()

cid10 = sus.read("aux_cid10")
causas = (
    obitos.join(cid10, left_on="causabas", right_on="codigo", how="left")
    .group_by("capitulo", "capitulo_descricao")
    .len("obitos")
    .sort("obitos", descending=True)
)
causas

## 10. AIH do SIH

O [perfil do SIH](https://raphaelfh.github.io/omnisus/sources/sih_aih_reduzida/) avisa
que o documento oficial não diz que cada linha é uma AIH distinta, nem como uma
internação longa se divide em AIH: compare linhas com números de AIH (`n_aih`) e chame
o resultado de AIH, não de internações. `morte = 1` tem o rótulo "Com óbito" no
dicionário.

O diagnóstico principal (`diag_princ`) junta com a mesma `aux_cid10` da seção 9.

In [ ]:
display(aih.select(linhas=pl.len(), aih_distintas=pl.col("n_aih").n_unique()))

por_capitulo = (
    aih.join(cid10, left_on="diag_princ", right_on="codigo", how="left")
    .group_by("capitulo", "capitulo_descricao")
    .agg(
        aih=pl.len(),
        aih_com_obito=(pl.col("morte") == 1).sum(),
        valor_total=pl.col("val_tot").sum().round(2),
    )
    .sort("aih", descending=True)
)
por_capitulo

## 11. Óbitos por 100 mil habitantes

`import_ibge_populacao` importa a população por município de uma edição explícita do
IBGE; `census=True` é o censo. Importar a mesma edição duas vezes faz a visão
`ibge_populacao` falhar, por isso a célula confere antes. O SIM usa 6 dígitos para o
município e o IBGE usa 7; a `aux_municipios` traz os dois (`codigo_6` e `codigo_ibge`),
e cada base junta pelo seu, sem inventar dígito. Os óbitos entram pelo município de
residência (`codmunres`). A tabela traz cada município da UF com população no censo; a
taxa da UF conta todos os óbitos carregados, inclusive os de município ignorado.

In [ ]:
with sus.LakeReader() as lake:
    edicoes = sus.cite(lake, dataset="ibge_populacao").publications
if not any(e["product"] == "census" and e["ano"] == 2022 for e in edicoes):
    sus.import_ibge_populacao(years=[2022], census=True)


populacao = sus.read("ibge_populacao").filter(pl.col("ano") == 2022)
codigo_uf = sus.read("aux_uf").filter(pl.col("sigla") == UF)["codigo_ibge"].item()
obitos_por_municipio = obitos.group_by("codmunres").len("obitos")
taxa = (
    sus.read("aux_municipios")
    .filter(pl.col("uf_codigo") == codigo_uf)
    .join(populacao, on="codigo_ibge")
    .join(obitos_por_municipio, left_on="codigo_6", right_on="codmunres", how="left")
    .select(pl.col("nome").alias("municipio"), "populacao", pl.col("obitos").fill_null(0))
    # Divide antes: os óbitos de uma capital vezes 100 mil estouram o inteiro.
    .with_columns(obitos_por_100_mil=(pl.col("obitos") / pl.col("populacao") * 100_000).round(1))
    .sort("obitos_por_100_mil", descending=True)
)
taxa_uf = 100_000 * obitos.height / taxa["populacao"].sum()
print(f"{UF}: {taxa_uf:.1f} óbitos de {ANO_SIM} por 100 mil habitantes do censo 2022")
sem_municipio = obitos.height - taxa["obitos"].sum()
if sem_municipio:
    print(f"{sem_municipio} desses óbitos não têm município da tabela e entram só no total.")
taxa

## 12. Citar

A citação nomeia cada arquivo do servidor, o SHA-256 dele e o snapshot do lake. Uma taxa
cita as duas bases: o SIM e a edição do IBGE. `cite` nomeia todas as publicações da base
no lake: se você trocou a UF ou os anos nesta sessão, ou usa um lake do Drive com outros
escopos, a citação traz também esses arquivos. Guarde a citação junto com o resultado;
veja [Reprodutibilidade](https://raphaelfh.github.io/omnisus/pesquisa/reprodutibilidade/).

In [ ]:
with sus.LakeReader() as lake:
    for base in ["sim_obitos", "sih_aih_reduzida", "ibge_populacao"]:
        print(sus.cite(lake, dataset=base).text, end="\n\n")

## Próximos passos

- Troque `UF` e os anos na seção 2, ou a base em `load` por um `name` da seção 3. Numa
  base `national`, tire `ufs`; numa `monthly`, passe `months=[...]`.
- Se usou o Drive, rode `drive.flush_and_unmount()` antes de fechar, para todos os
  arquivos chegarem lá.
- Os [notebooks de cada base](https://github.com/raphaelfh/omnisus/tree/main/notebooks)
  aprofundam cada sistema, e o
  [guia do pesquisador](https://raphaelfh.github.io/omnisus/pesquisa/) explica as
  escolhas.